# CSQE поверх RRF + Query2doc (CPU-only)

Реализация **Corpus-Steered Query Expansion** (Lei, Cao, Zhou, Shen, Yates — EACL 2024) точно по статье. Базовое LLM-расширение — **Query2doc**, как в оригинальной работе.

## Алгоритм

Для каждого запроса `q`:
1. PRF-источник: топ-K документов из `RRF(giga(q) + bm25(q))` — **на оригинальном запросе**.
2. Двухэтапный CSQE-промпт: LLM для каждого документа в топ-K решает «релевантен / нерелевантен» и для релевантных извлекает ключевые предложения.
3. Финальное расширение: `q + Query2doc(q) + key_sentences`.
4. Финальный поиск: `RRF-B` (вариант B) — `giga` получает оригинальный `q`, `bm25` получает финальное расширение.

**K = 10** (как в статье), 1 LLM-вызов на запрос с инструкцией обработать все 10 документов в JSON-формате.

## Что переиспользуется (без пересчёта)

Все тяжёлые операции (giga-embeddings, FAISS-поиск на корпусе) **уже сделаны** ранее, их результаты лежат в кэше. Этот ноутбук **не загружает GPU-модели** и работает только на CPU + DeepSeek API.

Обязательные кэш-файлы (проверяются при старте):
* `cache/corpus_proc_{ds}.json` — препроцессированный корпус для BM25
* `cache/giga_orig_run_{ds}.json` — run giga на оригинальных запросах (PRF-источник для giga-ветки)
* `expansions/__raw_query2doc_{ds}.json` — сырые Query2doc расширения

## Артефакты

* `expansions/rrf_query2doc_csqe_{ds}.json` — тройки `{qid, original, expanded, processed}`
* `runs/rrf_query2doc_csqe_{ds}.json` — top-100 для реранкера
* `cache/__raw_csqe_sentences_{ds}.json` — отдельный кэш извлечённых предложений (без Query2doc)
* `results/csqe_summary.csv` — финальная таблица с метриками

## 0. Установка зависимостей (CPU-only стек)

In [4]:
pip uninstall pytrec_eval -y

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install pytrec-eval-terrier

Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install -q datasets rank-bm25 tqdm openai pymorphy3 pandas nest_asyncio

Note: you may need to restart the kernel to use updated packages.


In [8]:
from __future__ import annotations

import os
import re
import sys
import json
import time
import asyncio
from collections import defaultdict
from pathlib import Path

import numpy as np
from datasets import load_dataset
from tqdm.auto import tqdm
import pytrec_eval
import pymorphy3
from rank_bm25 import BM25Okapi

# === Директории ===
ROOT = Path('.')
EXP_DIR = ROOT / 'expansions'
RUN_DIR = ROOT / 'runs'
RES_DIR = ROOT / 'results'
CACHE_DIR = ROOT / 'cache'
for d in (EXP_DIR, RUN_DIR, RES_DIR, CACHE_DIR):
    d.mkdir(exist_ok=True)

# === DeepSeek API ===
# DEEPSEEK_API_KEY_PLACEHOLDER'DEEPSEEK_API_KEY', '')
DEEPSEEK_API_KEY_PLACEHOLDER'

DEEPSEEK_MODEL = 'deepseek-chat'
DEEPSEEK_BASE_URL = 'https://api.deepseek.com'
DEEPSEEK_MAX_CONCURRENCY = 32

if not DEEPSEEK_API_KEY_PLACEHOLDER    print('WARNING: DEEPSEEK_API_KEY не задан в env')

# === Конфигурация CSQE ===
CSQE_PRF_DEPTH = 10           # K из статьи Lei et al.
CSQE_DOC_TRUNCATE = 1500      # символов на документ в промпте
CSQE_MAX_NEW_TOKENS = 1024    # ответ может быть длинным (до 10 предложений)
CSQE_TEMPERATURE = 0.0        # deterministic
CSQE_RRF_K = 60               # RRF parameter (стандарт)

## 1. Проверка кэша

Без обязательных файлов запускать нельзя — иначе пришлось бы пересчитывать giga-эмбеддинги, для чего нужен GPU. Этот ноутбук CPU-only по дизайну.

In [11]:
DATASET_NAMES = ['rus-scifact', 'rus-nfcorpus']

REQUIRED_FILES = []
for ds in DATASET_NAMES:
    REQUIRED_FILES += [
        CACHE_DIR / f'corpus_proc_{ds}.json',
        CACHE_DIR / f'giga_orig_run_{ds}.json',
        EXP_DIR / f'__raw_query2doc_{ds}.json',
    ]

missing = [str(p) for p in REQUIRED_FILES if not p.exists()]
if missing:
    print('ОШИБКА: отсутствуют обязательные файлы кэша:')
    for m in missing:
        print(f'  - {m}')
    print('\nЭтот ноутбук CPU-only и переиспользует кэш с GPU-машины.')
    print('Скопируй недостающие файлы перед запуском.')
    sys.exit(1)
else:
    print('Все обязательные файлы на месте, идём дальше.')

Все обязательные файлы на месте, идём дальше.


## 2. Загрузка датасетов

Корпус и queries — из HuggingFace. Колонки `processed_*` не используем (общая политика пайплайна — единый препроцессинг).

In [12]:
def load_beir_like(name: str, qrels_split: str = 'test'):
    corpus_ds = load_dataset(f'kaengreg/{name}', 'corpus')['train']
    queries_ds = load_dataset(f'kaengreg/{name}', 'queries')['train']
    qrels_ds = load_dataset(f'kaengreg/{name}-qrels', 'qrels')[qrels_split]

    corpus = {}
    for row in corpus_ds:
        corpus[str(row['_id'])] = {
            'title': row.get('title') or '',
            'text': row.get('text') or '',
        }

    qrels = defaultdict(dict)
    for row in qrels_ds:
        qrels[str(row['query-id'])][str(row['corpus-id'])] = int(row['score'])

    valid_qids = set(qrels.keys())
    queries = {}
    for row in queries_ds:
        qid = str(row['_id'])
        if qid in valid_qids:
            queries[qid] = {'text': row.get('text') or ''}
    qrels = {q: r for q, r in qrels.items() if q in queries}

    print(f'{name}: corpus={len(corpus)}, queries={len(queries)}, qrels={len(qrels)}')
    return corpus, queries, dict(qrels)


DATASETS = {ds: load_beir_like(ds, 'test') for ds in DATASET_NAMES}

rus-scifact: corpus=5183, queries=300, qrels=300
rus-nfcorpus: corpus=3633, queries=323, qrels=323


## 3. Препроцессинг (для bm25 и поля `processed`)

Тот же `preprocess()`, что и в основном пайплайне — для согласованности.

In [13]:
SOFT_RU_STOPWORDS = frozenset("""
и в во на с со от до из у к о об про над под при за по для
а но или либо да же ли бы б
я ты он она оно мы вы они меня тебя его её нас вас их
мне тебе ему ей нам вам им мной тобой нами вами ими
себя себе собой
это эта этот эти то та тот те
был была было были быть есть
только лишь уже ещё тоже также
вот ведь ну ах ой эх
если иначе чтобы хотя пока
""".split())

_morph = pymorphy3.MorphAnalyzer()
_TOKEN_RE = re.compile(r'[а-яёa-z0-9]+', re.IGNORECASE)
_lemma_cache: dict[str, str] = {}


def _lemma(token: str) -> str:
    cached = _lemma_cache.get(token)
    if cached is not None:
        return cached
    cached = _morph.parse(token)[0].normal_form
    _lemma_cache[token] = cached
    return cached


def preprocess(text: str) -> list[str]:
    text = text.lower()
    out = []
    for raw in _TOKEN_RE.findall(text):
        if len(raw) < 2:
            continue
        lemma = _lemma(raw)
        if lemma in SOFT_RU_STOPWORDS:
            continue
        out.append(lemma)
    return out


def preprocess_str(text: str) -> str:
    return ' '.join(preprocess(text))

## 4. Метрики (NDCG, Recall, MAP — все @10 и @100)

In [14]:
K_VALUES = [10, 100]


def evaluate_run(qrels: dict, run: dict, k_values=K_VALUES):
    metric_strs = set()
    for k in k_values:
        metric_strs.add(f'ndcg_cut.{k}')
        metric_strs.add(f'recall.{k}')
        metric_strs.add(f'map_cut.{k}')
    evaluator = pytrec_eval.RelevanceEvaluator(qrels, metric_strs)
    per_q = evaluator.evaluate(run)
    agg = defaultdict(list)
    for q, m in per_q.items():
        for k, v in m.items():
            agg[k].append(v)
    means = {k: float(np.mean(v)) for k, v in agg.items()}
    out = {}
    for k in k_values:
        out[f'NDCG@{k}'] = means.get(f'ndcg_cut_{k}', 0.0)
        out[f'Recall@{k}'] = means.get(f'recall_{k}', 0.0)
        out[f'MAP@{k}'] = means.get(f'map_cut_{k}', 0.0)
    return out

## 5. BM25-индекс (из кэша)

Корпус уже препроцессирован и лежит в `cache/corpus_proc_{ds}.json`. Просто строим из него BM25Okapi.

In [15]:
_BM25_INDEX_CACHE: dict[str, tuple] = {}


def get_bm25_index(dataset_name: str):
    if dataset_name in _BM25_INDEX_CACHE:
        return _BM25_INDEX_CACHE[dataset_name]
    cache = CACHE_DIR / f'corpus_proc_{dataset_name}.json'
    proc = json.loads(cache.read_text(encoding='utf-8'))
    doc_ids = list(proc.keys())
    print(f'  [BM25] building index for {dataset_name} ({len(doc_ids):,} docs)...')
    bm25 = BM25Okapi([proc[d] for d in doc_ids])
    _BM25_INDEX_CACHE[dataset_name] = (bm25, doc_ids)
    return bm25, doc_ids


def bm25_search(dataset_name: str, processed_queries: dict, k: int = 100) -> dict:
    """processed_queries: {qid: list[token]} — препроцессированные запросы."""
    bm25, doc_ids = get_bm25_index(dataset_name)
    run = {}
    for qid, q_tokens in tqdm(processed_queries.items(), desc=f'BM25 search {dataset_name}'):
        if not q_tokens:
            run[qid] = {}
            continue
        scores = bm25.get_scores(q_tokens)
        kk = min(k, len(scores))
        top_idx = np.argpartition(-scores, kk - 1)[:kk]
        top_idx = top_idx[np.argsort(-scores[top_idx])]
        run[qid] = {doc_ids[i]: float(scores[i]) for i in top_idx}
    return run

## 6. RRF (вариант B)

Сливаем `giga(оригинальный_запрос)` (из кэша) и `bm25(текущий_запрос)`.

In [16]:
def rrf_fuse(runs: list[dict], k_rrf: int = CSQE_RRF_K, top_k: int = 100) -> dict:
    qids = set().union(*[set(r.keys()) for r in runs])
    fused = {}
    for qid in qids:
        agg = defaultdict(float)
        for r in runs:
            if qid not in r:
                continue
            ranked = sorted(r[qid].items(), key=lambda x: -x[1])
            for rank, (did, _) in enumerate(ranked, start=1):
                agg[did] += 1.0 / (k_rrf + rank)
        top = sorted(agg.items(), key=lambda x: -x[1])[:top_k]
        fused[qid] = {d: float(s) for d, s in top}
    return fused


def load_giga_orig_run(dataset_name: str) -> dict:
    p = CACHE_DIR / f'giga_orig_run_{dataset_name}.json'
    return json.loads(p.read_text(encoding='utf-8'))

## 7. PRF-источник: RRF(giga + bm25) на оригинальных запросах

Согласно статье CSQE, PRF берётся на **оригинальном** запросе — независимо от LLM-расширения. Получаем топ-K документов из этого RRF и передаём их LLM.

In [17]:
def get_prf_top_k(dataset_name: str, queries: dict, k: int = CSQE_PRF_DEPTH) -> dict:
    """Возвращает {qid: [doc_id_1, ..., doc_id_k]} — top-K из RRF(giga_orig + bm25_orig).
    Кэшируется."""
    cache = CACHE_DIR / f'rrf_orig_run_{dataset_name}.json'
    if cache.exists():
        rrf_run = json.loads(cache.read_text(encoding='utf-8'))
    else:
        # bm25 на оригинальных (preprocess'нутых) запросах
        proc_q = {qid: preprocess(qd['text']) for qid, qd in queries.items()}
        bm25_orig = bm25_search(dataset_name, proc_q, k=100)
        # giga на оригинальных запросах — из кэша
        giga_orig = load_giga_orig_run(dataset_name)
        rrf_run = rrf_fuse([giga_orig, bm25_orig], k_rrf=CSQE_RRF_K, top_k=100)
        cache.write_text(json.dumps(rrf_run, ensure_ascii=False), encoding='utf-8')
        print(f'  Saved {cache}')
    # Берём топ-k для PRF
    out = {}
    for qid, ranked in rrf_run.items():
        sorted_docs = sorted(ranked.items(), key=lambda x: -x[1])[:k]
        out[qid] = [d for d, _ in sorted_docs]
    return out

## 8. DeepSeek API клиент

In [18]:
from openai import AsyncOpenAI


class DeepSeekClient:
    def __init__(self, api_key: str, base_url: str = DEEPSEEK_BASE_URL,
                 model: str = DEEPSEEK_MODEL,
                 max_concurrency: int = DEEPSEEK_MAX_CONCURRENCY):
        self.client = AsyncOpenAI(api_key=api_key, base_url=base_url)
        self.model = model
        self.sem = asyncio.Semaphore(max_concurrency)

    async def _one(self, messages, temperature=0.0, max_tokens=512, top_p=1.0):
        async with self.sem:
            for attempt in range(4):
                try:
                    resp = await self.client.chat.completions.create(
                        model=self.model,
                        messages=messages,
                        temperature=temperature,
                        top_p=top_p,
                        max_tokens=max_tokens,
                    )
                    return resp.choices[0].message.content or ''
                except Exception as e:
                    if attempt == 3:
                        print(f'  [deepseek] error after retries: {e}')
                        return ''
                    await asyncio.sleep(2 ** attempt)
        return ''

    async def batch(self, messages_list, **kwargs):
        tasks = [self._one(m, **kwargs) for m in messages_list]
        return await asyncio.gather(*tasks)


def _run_async(coro):
    """Запуск async coroutine из sync-кода (включая Jupyter)."""
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = None
    if loop and loop.is_running():
        import nest_asyncio
        nest_asyncio.apply()
        return loop.run_until_complete(coro)
    return asyncio.run(coro)


if DEEPSEEK_API_KEY_PLACEHOLDER    _ds_client = DeepSeekClient(DEEPSEEK_API_KEY)
else:
    _ds_client = None
    print('DeepSeek клиент не инициализирован — задайте DEEPSEEK_API_KEY')

## 9. CSQE: извлечение ключевых предложений

**Двухэтапный промпт по статье Lei et al.**: для каждого документа из top-K LLM сначала классифицирует его как «релевантный» или «нерелевантный», и только для релевантных извлекает ключевые предложения. Чтобы не делать K вызовов на запрос, упаковываем все K документов в один промпт и просим LLM вернуть JSON-массив.

Промпт адаптирован на русский (датасеты русскоязычные).

In [19]:
CSQE_SYSTEM = (
    'Ты — ассистент, помогающий с поиском релевантной информации в научных документах. '
    'Твоя задача — определить, какие документы релевантны поисковому запросу, '
    'и извлечь из них ключевые предложения, которые отвечают на запрос.'
)

CSQE_USER_TEMPLATE = (
    'ЗАПРОС: {QUERY}\n\n'
    'Ниже приведены документы, найденные поисковой системой. Для каждого документа:\n'
    '1. Определи, релевантен ли он запросу (да / нет).\n'
    '2. Если релевантен — извлеки 1–3 ключевых предложения, которые наиболее '
    'точно отвечают на запрос. Бери предложения дословно из документа.\n'
    '3. Если нерелевантен — оставь поле "sentences" пустым.\n\n'
    '{DOCS}\n\n'
    'Ответ верни СТРОГО в формате JSON-массива (без пояснений и markdown):\n'
    '[\n'
    '  {{"doc_id": 1, "relevant": true, "sentences": ["предложение 1", "предложение 2"]}},\n'
    '  {{"doc_id": 2, "relevant": false, "sentences": []}},\n'
    '  ...\n'
    ']'
)


def _format_docs(doc_ids, corpus, truncate=CSQE_DOC_TRUNCATE):
    parts = []
    for i, did in enumerate(doc_ids, start=1):
        d = corpus.get(did, {})
        txt = ((d.get('title', '') + ' ' + d.get('text', '')).strip())[:truncate]
        parts.append(f'[Документ {i}]\n{txt}')
    return '\n\n'.join(parts)


_JSON_BLOCK_RE = re.compile(r'\[.*\]', re.DOTALL)


def _parse_csqe_response(raw: str) -> list[str]:
    """Достаём список ключевых предложений (плоский) из ответа LLM."""
    if not raw:
        return []
    s = raw.strip()
    s = re.sub(r'^```(?:json)?\s*', '', s)
    s = re.sub(r'\s*```$', '', s)
    m = _JSON_BLOCK_RE.search(s)
    if m:
        s = m.group(0)
    try:
        data = json.loads(s)
    except Exception:
        return []
    out = []
    if not isinstance(data, list):
        return []
    for item in data:
        if not isinstance(item, dict):
            continue
        if not item.get('relevant'):
            continue
        sents = item.get('sentences', []) or []
        for s in sents:
            if isinstance(s, str) and s.strip():
                out.append(s.strip())
    return out


async def _csqe_batch(qids, qtexts, prf_top_k, corpus):
    """Батч CSQE-вызовов через DeepSeek API."""
    messages_list = []
    for qid, qt in zip(qids, qtexts):
        doc_ids = prf_top_k.get(qid, [])
        docs_str = _format_docs(doc_ids, corpus)
        user = CSQE_USER_TEMPLATE.replace('{QUERY}', qt).replace('{DOCS}', docs_str)
        messages_list.append([
            {'role': 'system', 'content': CSQE_SYSTEM},
            {'role': 'user',   'content': user},
        ])
    raws = await _ds_client.batch(
        messages_list,
        temperature=CSQE_TEMPERATURE,
        max_tokens=CSQE_MAX_NEW_TOKENS,
    )
    return raws


def csqe_extract_sentences(dataset_name: str, queries: dict, corpus: dict,
                           prf_top_k: dict) -> dict:
    """Возвращает {qid: list[str]} — извлечённые ключевые предложения.
    Кэшируется на диске в cache/__raw_csqe_sentences_{ds}.json."""
    cache = CACHE_DIR / f'__raw_csqe_sentences_{dataset_name}.json'
    if cache.exists():
        print(f'  [CSQE] cached: {cache}')
        return json.loads(cache.read_text(encoding='utf-8'))
    if _ds_client is None:
        raise RuntimeError('DeepSeek API key не задан, и кэша CSQE нет')

    qids = list(queries.keys())
    qtexts = [queries[q]['text'] for q in qids]
    print(f'  [CSQE] {len(qids):,} LLM calls (DeepSeek API) ...')

    raws = _run_async(_csqe_batch(qids, qtexts, prf_top_k, corpus))
    out = {qid: _parse_csqe_response(raw) for qid, raw in zip(qids, raws)}

    n_with = sum(1 for v in out.values() if v)
    n_sents = sum(len(v) for v in out.values())
    print(f'  [CSQE] queries with extracted sentences: {n_with}/{len(out)}, '
          f'total sentences: {n_sents}, avg per query: {n_sents/len(out):.2f}')

    cache.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding='utf-8')
    return out

## 10. Финальное расширение и поиск

Для каждого запроса:
* `expanded = original + ' ' + query2doc + ' ' + ' '.join(key_sentences)`
* `processed = preprocess(expanded)` → подаётся в bm25
* giga в RRF-B получает оригинальный запрос (через кэш `giga_orig_run`)
* RRF фьюзит giga_orig + bm25(csqe)

In [20]:
def build_csqe_records(queries: dict, raw_q2d: dict, csqe_sents: dict) -> list[dict]:
    """Строит тройки {qid, original, expanded, processed}."""
    records = []
    for qid, qd in queries.items():
        original = qd['text']
        q2d = (raw_q2d.get(qid) or '').strip()
        sents = csqe_sents.get(qid, []) or []
        sents_str = ' '.join(sents)
        parts = [original]
        if q2d:
            parts.append(q2d)
        if sents_str:
            parts.append(sents_str)
        expanded = ' '.join(parts).strip()
        processed = preprocess_str(expanded)
        records.append({
            'qid': qid,
            'original': original,
            'expanded': expanded,
            'processed': processed,
        })
    return records

## 11. Прогон на двух датасетах

In [21]:
all_metrics = []

for ds_name, (corpus, queries, qrels) in DATASETS.items():
    print(f'\n{"=" * 70}\nDataset: {ds_name}\n{"=" * 70}')

    # 1) Загружаем сырые Query2doc расширения
    raw_q2d_path = EXP_DIR / f'__raw_query2doc_{ds_name}.json'
    raw_q2d = json.loads(raw_q2d_path.read_text(encoding='utf-8'))
    print(f'  Loaded {len(raw_q2d)} Query2doc expansions from {raw_q2d_path.name}')

    # 2) PRF-источник: top-K из RRF(giga_orig + bm25_orig)
    print('  Computing PRF source: RRF(giga_orig + bm25_orig) ...')
    prf_top_k = get_prf_top_k(ds_name, queries, k=CSQE_PRF_DEPTH)

    # 3) CSQE: извлечь ключевые предложения через LLM
    print('  Running CSQE ...')
    t0 = time.time()
    csqe_sents = csqe_extract_sentences(ds_name, queries, corpus, prf_top_k)
    print(f'  CSQE done in {time.time() - t0:.1f}s')

    # 4) Собираем расширения и сохраняем
    records = build_csqe_records(queries, raw_q2d, csqe_sents)
    exp_path = EXP_DIR / f'rrf_query2doc_csqe_{ds_name}.json'
    exp_path.write_text(json.dumps(records, ensure_ascii=False, indent=2),
                        encoding='utf-8')
    print(f'  Saved expansions: {exp_path}')

    # 5) BM25 search на CSQE-расширенных запросах
    proc_queries = {r['qid']: r['processed'].split() for r in records}
    print('  BM25 search on CSQE-expanded queries ...')
    bm25_run = bm25_search(ds_name, proc_queries, k=100)

    # 6) Финальный RRF: giga_orig + bm25(CSQE)
    giga_orig_run = load_giga_orig_run(ds_name)
    final_run = rrf_fuse([giga_orig_run, bm25_run], k_rrf=CSQE_RRF_K, top_k=100)

    # 7) Сохраняем run
    run_path = RUN_DIR / f'rrf_query2doc_csqe_{ds_name}.json'
    run_path.write_text(json.dumps(final_run, ensure_ascii=False), encoding='utf-8')
    print(f'  Saved run: {run_path}')

    # 8) Метрики
    m = evaluate_run(qrels, final_run)
    all_metrics.append({
        'dataset': ds_name, 'retriever': 'rrf', 'qe': 'query2doc_csqe', **m
    })
    print(f'\n  RESULT  rrf + query2doc + csqe  on  {ds_name}:')
    for k, v in m.items():
        print(f'    {k:12s} = {v:.4f}')


Dataset: rus-scifact
  Loaded 300 Query2doc expansions from __raw_query2doc_rus-scifact.json
  Computing PRF source: RRF(giga_orig + bm25_orig) ...
  [BM25] building index for rus-scifact (5,183 docs)...


BM25 search rus-scifact:   0%|          | 0/300 [00:00<?, ?it/s]

  Saved cache\rrf_orig_run_rus-scifact.json
  Running CSQE ...
  [CSQE] 300 LLM calls (DeepSeek API) ...
  [CSQE] queries with extracted sentences: 220/300, total sentences: 533, avg per query: 1.78
  CSQE done in 46.4s
  Saved expansions: expansions\rrf_query2doc_csqe_rus-scifact.json
  BM25 search on CSQE-expanded queries ...


BM25 search rus-scifact:   0%|          | 0/300 [00:00<?, ?it/s]

  Saved run: runs\rrf_query2doc_csqe_rus-scifact.json

  RESULT  rrf + query2doc + csqe  on  rus-scifact:
    NDCG@10      = 0.7645
    Recall@10    = 0.8747
    MAP@10       = 0.7240
    NDCG@100     = 0.7855
    Recall@100   = 0.9700
    MAP@100      = 0.7286

Dataset: rus-nfcorpus
  Loaded 323 Query2doc expansions from __raw_query2doc_rus-nfcorpus.json
  Computing PRF source: RRF(giga_orig + bm25_orig) ...
  [BM25] building index for rus-nfcorpus (3,633 docs)...


BM25 search rus-nfcorpus:   0%|          | 0/323 [00:00<?, ?it/s]

  Saved cache\rrf_orig_run_rus-nfcorpus.json
  Running CSQE ...
  [CSQE] 323 LLM calls (DeepSeek API) ...
  [CSQE] queries with extracted sentences: 236/323, total sentences: 1424, avg per query: 4.41
  CSQE done in 73.9s
  Saved expansions: expansions\rrf_query2doc_csqe_rus-nfcorpus.json
  BM25 search on CSQE-expanded queries ...


BM25 search rus-nfcorpus:   0%|          | 0/323 [00:00<?, ?it/s]

  Saved run: runs\rrf_query2doc_csqe_rus-nfcorpus.json

  RESULT  rrf + query2doc + csqe  on  rus-nfcorpus:
    NDCG@10      = 0.3941
    Recall@10    = 0.1935
    MAP@10       = 0.1525
    NDCG@100     = 0.3614
    Recall@100   = 0.3649
    MAP@100      = 0.1933


## 12. Сводная таблица

In [22]:
import pandas as pd

df = pd.DataFrame(all_metrics)
metric_cols = ['NDCG@10', 'NDCG@100', 'Recall@10', 'Recall@100', 'MAP@10', 'MAP@100']
for col in metric_cols:
    df[col] = df[col].round(4)
df = df[['dataset', 'retriever', 'qe'] + metric_cols]
df.sort_values(['dataset', 'retriever', 'qe'], inplace=True)
df.to_csv(RES_DIR / 'csqe_summary.csv', index=False)
df

,dataset,retriever,qe,NDCG@10,NDCG@100,Recall@10,Recall@100,MAP@10,MAP@100
1,rus-nfcorpus,rrf,query2doc_csqe,0.3941,0.3614,0.1935,0.3649,0.1525,0.1933
0,rus-scifact,rrf,query2doc_csqe,0.7645,0.7855,0.8747,0.9700,0.7240,0.7286


## 13. (Опционально) сравнение с baseline rrf+query2doc

Если в `runs/rrf_query2doc_{ds}.json` лежат baseline-runs от предыдущего эксперимента, можем сравнить метрики бок-о-бок.

In [23]:
compare_rows = []
for ds_name, (corpus, queries, qrels) in DATASETS.items():
    baseline_path = RUN_DIR / f'rrf_query2doc_{ds_name}.json'
    if not baseline_path.exists():
        print(f'  Baseline not found: {baseline_path}, skipping {ds_name}')
        continue
    base_run = json.loads(baseline_path.read_text(encoding='utf-8'))
    base_m = evaluate_run(qrels, base_run)
    csqe_m = next(m for m in all_metrics if m['dataset'] == ds_name)
    for metric in ['NDCG@10', 'NDCG@100', 'Recall@10', 'Recall@100', 'MAP@10', 'MAP@100']:
        compare_rows.append({
            'dataset': ds_name,
            'metric': metric,
            'baseline (rrf+q2d)': round(base_m[metric], 4),
            'rrf+q2d+csqe': round(csqe_m[metric], 4),
            'delta': round(csqe_m[metric] - base_m[metric], 4),
        })

if compare_rows:
    cmp_df = pd.DataFrame(compare_rows)
    cmp_df.to_csv(RES_DIR / 'csqe_vs_baseline.csv', index=False)
    print()
    print(cmp_df.to_string(index=False))


     dataset     metric  baseline (rrf+q2d)  rrf+q2d+csqe  delta
 rus-scifact    NDCG@10              0.7437        0.7645 0.0208
 rus-scifact   NDCG@100              0.7652        0.7855 0.0202
 rus-scifact  Recall@10              0.8708        0.8747 0.0039
 rus-scifact Recall@100              0.9667        0.9700 0.0033
 rus-scifact     MAP@10              0.6970        0.7240 0.0270
 rus-scifact    MAP@100              0.7022        0.7286 0.0264
rus-nfcorpus    NDCG@10              0.3691        0.3941 0.0250
rus-nfcorpus   NDCG@100              0.3389        0.3614 0.0225
rus-nfcorpus  Recall@10              0.1807        0.1935 0.0128
rus-nfcorpus Recall@100              0.3382        0.3649 0.0267
rus-nfcorpus     MAP@10              0.1424        0.1525 0.0101
rus-nfcorpus    MAP@100              0.1782        0.1933 0.0151


In [24]:
compare_rows = []
for ds_name, (corpus, queries, qrels) in DATASETS.items():
    baseline_path = RUN_DIR / f'rrf_word2passage_{ds_name}.json'
    if not baseline_path.exists():
        print(f'  Baseline not found: {baseline_path}, skipping {ds_name}')
        continue
    base_run = json.loads(baseline_path.read_text(encoding='utf-8'))
    base_m = evaluate_run(qrels, base_run)
    csqe_m = next(m for m in all_metrics if m['dataset'] == ds_name)
    for metric in ['NDCG@10', 'NDCG@100', 'Recall@10', 'Recall@100', 'MAP@10', 'MAP@100']:
        compare_rows.append({
            'dataset': ds_name,
            'metric': metric,
            'baseline (rrf+w2p)': round(base_m[metric], 4),
            'rrf+q2d+csqe': round(csqe_m[metric], 4),
            'delta': round(csqe_m[metric] - base_m[metric], 4),
        })

if compare_rows:
    cmp_df = pd.DataFrame(compare_rows)
    cmp_df.to_csv(RES_DIR / 'csqe_vs_baseline.csv', index=False)
    print()
    print(cmp_df.to_string(index=False))


     dataset     metric  baseline (rrf+w2p)  rrf+q2d+csqe   delta
 rus-scifact    NDCG@10              0.7602        0.7645  0.0043
 rus-scifact   NDCG@100              0.7800        0.7855  0.0055
 rus-scifact  Recall@10              0.8748        0.8747 -0.0001
 rus-scifact Recall@100              0.9633        0.9700  0.0067
 rus-scifact     MAP@10              0.7190        0.7240  0.0050
 rus-scifact    MAP@100              0.7235        0.7286  0.0051
rus-nfcorpus    NDCG@10              0.3683        0.3941  0.0257
rus-nfcorpus   NDCG@100              0.3364        0.3614  0.0250
rus-nfcorpus  Recall@10              0.1786        0.1935  0.0148
rus-nfcorpus Recall@100              0.3346        0.3649  0.0304
rus-nfcorpus     MAP@10              0.1375        0.1525  0.0150
rus-nfcorpus    MAP@100              0.1745        0.1933  0.0188
